# 00 — Data

Download GSM8K + SVAMP once, clean, split, save to `OUT_DIR`. Other notebooks read from there.

On Kaggle: run → Save Version → in other notebooks *Add Input* this notebook → `/kaggle/input/<slug>/data/`.

In [ ]:
from pathlib import Path
import pandas as pd
from datasets import load_dataset

N_VAL = 500
OUT_DIR = Path("/kaggle/working/data") if Path("/kaggle/working").exists() else Path("../outputs/data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Load

In [ ]:
gsm_train = load_dataset("openai/gsm8k", "main", split="train").to_pandas()
gsm_test  = load_dataset("openai/gsm8k", "main", split="test").to_pandas()
svamp = pd.concat([load_dataset("ChilleD/SVAMP", split=s).to_pandas() for s in ["train", "test"]], ignore_index=True)

print(len(gsm_train), len(gsm_test), len(svamp))
print(gsm_train.answer[0])

## Clean

- Remove `<<48/2=24>>` calculator notes (we have no calculator).
- gold = text after `####`, commas removed.
- SVAMP: question = Body + Question, all 1000 problems used as eval.

In [ ]:
for df in (gsm_train, gsm_test):
    df["solution"] = df.answer.str.replace(r"<<[^>]*>>", "", regex=True)
    df["gold"] = df.answer.str.split("####").str[-1].str.strip().str.replace(",", "")

svamp["question"] = svamp.Body.str.strip() + " " + svamp.Question.str.strip()
svamp["gold"] = svamp.Answer.astype(float).map(lambda x: str(int(x)) if x.is_integer() else str(x))

print(gsm_train.solution[0])

## Train / val split

In [ ]:
gsm_train = gsm_train.sample(frac=1, random_state=42).reset_index(drop=True)
gsm_val, gsm_train = gsm_train[:N_VAL], gsm_train[N_VAL:]

splits = {
    "gsm8k_train": gsm_train[["question", "solution", "gold"]],
    "gsm8k_val":   gsm_val[["question", "solution", "gold"]],
    "gsm8k_test":  gsm_test[["question", "solution", "gold"]],
    "svamp_test":  svamp[["question", "gold", "Type"]],
}
{k: len(v) for k, v in splits.items()}

## EDA

In [ ]:
for name, df in splits.items():
    g = df.gold.astype(float)
    print(f"{name:12s} q_words={df.question.str.split().str.len().median():.0f}  "
          f"steps={df.solution.str.count(chr(10)).median() if 'solution' in df else '-'}  "
          f"neg={(g < 0).sum()}  decimal={(g % 1 != 0).sum()}  >=1000={(g >= 1000).sum()}")

In [ ]:
# overlap between sets (should be 0)
qs = {k: set(v.question.str.lower()) for k, v in splits.items()}
names = list(qs)
for i, a in enumerate(names):
    for b in names[i + 1:]:
        print(a, b, len(qs[a] & qs[b]))

In [ ]:
# token lengths with the policy tokenizer -> pick max_new_tokens / max_seq_len
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
sol_tok = gsm_train.solution.map(lambda s: len(tok(s).input_ids))
sol_tok.describe(percentiles=[.5, .9, .99])

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
sol_tok.hist(bins=50, ax=ax[0]); ax[0].set_title("gsm8k train solution tokens")
svamp.Type.value_counts().plot.bar(ax=ax[1], title="svamp types")
plt.tight_layout()

## Save

In [ ]:
for name, df in splits.items():
    df.to_json(OUT_DIR / f"{name}.jsonl", orient="records", lines=True)
print(list(OUT_DIR.iterdir()))

# in other notebooks:
# df = pd.read_json(DATA_DIR / "gsm8k_train.jsonl", lines=True, dtype={"gold": str})